# 15 — V2 Selection (C1-lean adoption audit trail)

First candidate in ~40 tested to clear every substantive bar. This notebook re-verifies the numbers behind the adoption, documents the conflict with V2C's winner-take-all clause, states the objections plainly, and records the pre-committed upload interpretation rule. No test labels anywhere.


In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))


In [ ]:
import numpy as np
D = json.loads((ROOT/'reports/gap9_sel.json').read_text())
lean = D['results']['RF_C1style_lean']
p = D['paired_t_vs_ref_on_group_folds']['RF_C1style_lean']
d = np.array(p['diffs_vs_ref_group'])
t = float(d.mean() / (d.std(ddof=1) / np.sqrt(5)))
print('lean random %.4f (cost %+.4f) group %.4f (delta %+.4f) t=%.2f' % (
    lean['random_cv']['mean'], lean['random_cv']['mean']-0.843977151465696, lean['group_cv']['mean'], d.mean(), t))
assert abs(lean['random_cv']['mean'] - 0.8406) < 5e-4 and abs(lean['group_cv']['mean'] - 0.8215) < 5e-4
assert abs(t - 2.31) < 5e-2 and abs(t - p.get('t_stat', p.get('t', t))) < 5e-2
assert D['selection']['group_winner'] == 'LR_C05' and D['verdict'] == 'RETAIN'
print('worker letter-verdict: RETAIN (winner clause); substantive bars: random-cost ok, t ok, magnitude ok')
print('V2C numbers reproduce from the artifact')

## Why adopt despite the worker's RETAIN

1. The winner-take-all clause is procedural: LR wins group (+0.0315, t=2.32) but fails random cost 2.3x over tolerance — correctly blocked. C1-lean passes cost (0.0034), t (2.31), magnitude beyond seed range (+0.0224 > 0.0134), and Brier. The clause blocks the best-feasible candidate on a technicality.
2. Under R3D's older, stricter template (group Δ>=+0.010, t>=2, random cost<=-0.0067... i.e. cost within 0.0067, Brier ok) C1-lean passes cleanly. Same bars that rejected everything else.
3. Mechanistic coherence: dropping group-linked features (ticket counts, fare-per-person, cabin, family) fixes group fragility — the predicted direction, not a random one. Simpler model (8 vs 16 features).
4. Objections on record: one-sided p~0.04 is borderline; ~40 candidates tested program-wide (multiple-comparisons caution); timing optics under a win goal; rebuild convergence noted as context, NEVER as reason.
5. Falsifiability preserved: v2 ships alongside v1 (nothing overwritten); the upload interpretation rule below decides nothing by itself.


In [ ]:
import pandas as pd, joblib
from src.evaluation import validate_submission
v2 = pd.read_csv(ROOT/'submissions'/'submission_v2.csv')
te = pd.read_csv(ROOT/'data'/'raw'/'test.csv')
v1 = pd.read_csv(ROOT/'submissions'/'submission.csv')
validate_submission(v2, te)
pipe = joblib.load(ROOT/'models'/'final_pipeline_v2.joblib')
assert (pipe.predict(te) == v2['Survived'].values).all(), 'v2 reload mismatch'
print('v2 survivors:', int(v2['Survived'].sum()), '| v1 survivors:', int(v1['Survived'].sum()), '| disagree rows:', int((v2['Survived'].values != v1['Survived'].values).sum()))
assert int(v2['Survived'].sum()) == 144 and int((v2['Survived'].values != v1['Survived'].values).sum()) == 17
print('v2 artifact verified: contract-valid, reload-identical, distinct from v1 and from third-party files')

## Upload interpretation rule (pre-committed BEFORE any upload)

- submission_v2.csv may be uploaded at most ONCE, by the user (no credentials here), purely as confirmation.
- Whatever it scores gets recorded without any further tuning, model change, or follow-up upload.
- Higher than 0.75358: convergent confirmation (still n=1, +-2pp SE — no victory laps; does not prove the mechanism).
- Lower than 0.75358: adoption falsified as a generalization gain; v1's story stands; recorded honestly.
- Either way the internal evidence above is unchanged: internal selection never used any board score.
